<a href="https://colab.research.google.com/github/Hkasberg23/BPA-NWM/blob/main/PROJECT1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install rdkit
%pip install scikit-learn
%pip install scipy
%pip install pandas
%pip install numpy
%pip install matplotlib

import warnings
from collections import defaultdict

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore")

DATA_URL = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/Lipophilicity.csv"
SEED = 42
ALPHA = 0.05

EXPERT_DESCRIPTORS = {
    "MolLogP": "Crippen logP: direct estimate of neutral-form partitioning.",
    "MolMR": "Molar refractivity (polarizability): dispersion forces favour octanol.",
    "FractionCSP3": "Saturation: flat aromatic molecules are usually more lipophilic.",
    "TPSA": "Polar surface area: polar atoms are strongly solvated by water.",
    "NumHDonors": "H-bond donors: strong water interactions.",
    "NumHAcceptors": "H-bond acceptors: strong water interactions.",
    "NOCount": "N + O count: crude polarity measure.",
    "NHOHCount": "N-H / O-H groups: strongest H-bond donors.",
    "NumHeteroatoms": "Non-carbon atoms: overall polarity.",
    "MolWt": "Larger molecules pay a bigger cavity cost in water.",
    "HeavyAtomCount": "Size measure insensitive to heavy-halogen mass.",
    "LabuteASA": "Accessible surface area: hydrophobic contact area.",
    "NumRotatableBonds": "Flexibility: hides/exposes polar groups.",
    "RingCount": "Rigid, mostly hydrophobic scaffolds.",
    "NumAromaticRings": "Aromatic pi-systems add lipophilicity.",
    "NumAliphaticRings": "Saturated rings add hydrophobic bulk.",
    "NumAromaticHeterocycles": "Pyridine-type N / O / S in rings are polar acceptors.",
    "BertzCT": "Topological complexity: size/branching proxy.",
    "fr_halogen": "Halogens add hydrophobicity.",
    "fr_amide": "Amides: polar donor + acceptor.",
    "fr_ester": "Esters: polar acceptors.",
    "fr_ether": "Ethers: weak acceptors.",
    "fr_Ar_OH": "Phenols: donors, weakly acidic.",
    "fr_sulfonamd": "Sulfonamides: very polar, weakly acidic.",
    "fr_benzene": "Benzene rings: lipophilic surface.",
    "fr_COO": "Carboxylic acids (aliphatic): anionic at pH 7.4.",
    "fr_COO2": "Carboxylic acids (aromatic): anionic at pH 7.4.",
    "fr_tetrazole": "Tetrazoles: carboxylic-acid mimics, anionic at pH 7.4.",
    "fr_NH0": "Tertiary aliphatic amines: often protonated at pH 7.4.",
    "fr_NH1": "Secondary aliphatic amines: often protonated at pH 7.4.",
    "fr_NH2": "Primary aliphatic amines: often protonated at pH 7.4.",
    "fr_amidine": "Amidines: strongly basic, cationic.",
    "fr_guanido": "Guanidines: strongly basic, cationic.",
    "fr_quatN": "Quaternary ammonium: permanent positive charge.",
}

# =====================================================================================
# 3. DATA LOADING AND SCAFFOLD SPLIT
# =====================================================================================
def load_data(path_or_url=DATA_URL):
    """Read the CSV (columns: CMPD_CHEMBLID, exp, smiles) and parse SMILES with RDKit."""
    df = pd.read_csv(path_or_url)
    df["mol"] = df["smiles"].apply(Chem.MolFromSmiles)      # None if SMILES is invalid
    return df.dropna(subset=["mol", "exp"]).reset_index(drop=True)


def scaffold_split(mols, test_frac=0.15):

    scafs = [MurckoScaffold.MurckoScaffoldSmiles(mol=m, includeChirality=False) for m in mols]
    groups = defaultdict(list)
    for i, s in enumerate(scafs):
        groups[s].append(i)

    ordered = sorted(groups.values(), key=lambda g: (len(g), g[0]), reverse=True)
    n_train = (1 - test_frac) * len(mols)
    train, test = [], []
    for g in ordered:
        (train if len(train) + len(g) <= n_train else test).extend(g)
    return np.array(train), np.array(test), np.array(scafs)

# =====================================================================================
# 4. FEATURIZATION: expert descriptors + Morgan count fingerprints
# =====================================================================================
_fp_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)


def featurize(mols):
    """Return (descriptor_table, fingerprint_table) as pandas DataFrames."""
    desc_fns = dict(Descriptors.descList)                    # name -> function
    names = [n for n in EXPERT_DESCRIPTORS if n in desc_fns]
    missing = sorted(set(EXPERT_DESCRIPTORS) - set(names))
    if missing:
        print(f"[warn] descriptors missing in this RDKit version, skipped: {missing}")

    desc = pd.DataFrame([[desc_fns[n](m) for n in names] for m in mols], columns=names)
    desc = desc.replace([np.inf, -np.inf], np.nan).fillna(0.0)


    fp = np.array([_fp_gen.GetCountFingerprintAsNumPy(m) for m in mols], dtype=float)
    fp = pd.DataFrame(fp, columns=[f"fp_{i}" for i in range(fp.shape[1])])
    return desc, fp

# =====================================================================================
# 5. REDUNDANCY ANALYSIS
# =====================================================================================
def drop_redundant(X_train, threshold=0.90):

    X = X_train.loc[:, X_train.std() > 0]
    corr = np.abs(np.nan_to_num(np.corrcoef(X.values, rowvar=False)))
    keep = []
    for j in range(corr.shape[0]):
        if all(corr[j, k] <= threshold for k in keep):
            keep.append(j)
    return list(X.columns[keep])

# =====================================================================================
# 6. EVALUATION HELPERS
# =====================================================================================
def report(name, y, p):
    rmse = np.sqrt(mean_squared_error(y, p))
    print(f"{name:<26} RMSE={rmse:.3f}  MAE={mean_absolute_error(y, p):.3f}  R2={r2_score(y, p):.3f}")


def normality(x, label):
    """Shapiro-Wilk and D'Agostino-Pearson tests (H0: data are normally distributed)."""
    p_sw = stats.shapiro(x)[1]
    p_dp = stats.normaltest(x)[1]
    verdict = "consistent with normal" if min(p_sw, p_dp) > ALPHA else "NOT normal"
    print(f"  {label:<22} Shapiro-Wilk p={p_sw:.3g} | D'Agostino-Pearson p={p_dp:.3g} -> {verdict}")
    return min(p_sw, p_dp) > ALPHA

def main():

    df = load_data()
    desc, fp = featurize(df["mol"])
    X_all = pd.concat([desc, fp], axis=1)
    y = df["exp"].values
    print(f"{len(df)} molecules | {desc.shape[1]} expert descriptors + {fp.shape[1]} fingerprint bits\n")

    tr, te, scafs = scaffold_split(df["mol"].tolist(), test_frac=0.15)
    print(f"scaffold split: train={len(tr)}  test={len(te)}\n")
    ytr, yte = y[tr], y[te]


    print("=== Statistical analysis (training set) ===")
    print("Normality of the target:")
    normality(ytr, "logD")

    desc_tr = desc.iloc[tr]
    n_normal = sum(
        1 for c in desc_tr.columns
        if desc_tr[c].std() > 0 and min(stats.shapiro(desc_tr[c])[1], stats.normaltest(desc_tr[c])[1]) > ALPHA
    )
    print(f"Descriptors passing both normality tests: {n_normal}/{desc_tr.shape[1]}")
    print("  -> most are not normal, so Spearman rank correlation is used to rank them.\n")

    rho = {}
    for c in desc_tr.columns:
        if desc_tr[c].std() > 0:
            rho[c] = stats.spearmanr(desc_tr[c], ytr)[0]
    rho = pd.Series(rho).sort_values(key=np.abs, ascending=False)
    print("Top 10 descriptors by |Spearman rho| with logD:")
    print(rho.head(10).round(3).to_string(), "\n")

    acid = (desc_tr["fr_COO"] + desc_tr["fr_COO2"]) > 0
    t, p = stats.ttest_ind(ytr[acid.values], ytr[~acid.values], equal_var=False)
    print(f"Welch t-test, logD of acids (n={acid.sum()}, mean={ytr[acid.values].mean():.2f}) vs "
          f"non-acids (n={(~acid).sum()}, mean={ytr[~acid.values].mean():.2f}): t={t:.2f}, p={p:.3g}\n")

    kept_cols = drop_redundant(X_all.iloc[tr], threshold=0.90)
    Xtr = X_all.iloc[tr][kept_cols].values
    Xte = X_all.iloc[te][kept_cols].values
    print(f"After constant + redundancy (|r|>0.90) filtering: {len(kept_cols)} of {X_all.shape[1]} features\n")

    groups = pd.factorize(scafs[tr])[0]
    cv = list(GroupKFold(n_splits=5).split(Xtr, ytr, groups))

    ks = [k for k in (30, 100, 200) if k <= len(kept_cols)]
    searches = {
        "SVR (RBF)": (
            Pipeline([("scale", StandardScaler()),
                      ("select", SelectKBest(f_regression)),
                      ("model", SVR(kernel="rbf"))]),
            {"select__k": ks,
             "model__C": [1, 3, 10],
             "model__gamma": ["scale", 0.003, 0.01],
             "model__epsilon": [0.1, 0.2]},
        ),
        "ExtraTrees": (
            Pipeline([("scale", StandardScaler()),
                      ("select", SelectKBest(f_regression)),
                      ("model", ExtraTreesRegressor(n_jobs=-1, random_state=SEED))]),
            {"select__k": ks,
             "model__n_estimators": [300],
             "model__max_features": [0.3, 0.6, 1.0],
             "model__min_samples_leaf": [1, 2]},
        ),
    }

    print("=== Test-set performance (held-out scaffolds) ===")
    base = DummyRegressor(strategy="mean").fit(Xtr, ytr)
    base_pred = base.predict(Xte)
    report("Baseline (mean)", yte, base_pred)

    fitted, preds, train_preds = {}, {}, {}
    for name, (pipe, grid) in searches.items():
        gs = GridSearchCV(pipe, grid, cv=cv, scoring="neg_root_mean_squared_error",
                          n_jobs=-1, refit=True)
        gs.fit(Xtr, ytr)
        fitted[name], preds[name] = gs, gs.predict(Xte)
        train_preds[name] = gs.predict(Xtr)
        report(name + " [TRAIN]", ytr, train_preds[name])
        report(name + " [TEST]", yte, preds[name])
        print(f"   best CV RMSE={-gs.best_score_:.3f} | params={gs.best_params_}")

        res = pd.DataFrame(gs.cv_results_).sort_values("rank_test_score").head(5)
        for _, row in res.iterrows():
            print(f"     rank {int(row['rank_test_score'])}: CV RMSE={-row['mean_test_score']:.3f} "
                  f"+/- {row['std_test_score']:.3f}  {row['params']}")
        pd.DataFrame(gs.cv_results_).to_csv(f"gridsearch_{name.split()[0]}.csv", index=False)


    best_name = min(fitted, key=lambda n: -fitted[n].best_score_)
    pred = preds[best_name]
    print(f"\nBest model by grouped-CV RMSE: {best_name}")


    support = fitted[best_name].best_estimator_.named_steps["select"].get_support()
    final_feats = np.array(kept_cols)[support]
    print(f"\nFeatures kept by SelectKBest ({len(final_feats)}): "
          f"{[f for f in final_feats if not f.startswith('fp_')]} + "
          f"{sum(f.startswith('fp_') for f in final_feats)} fingerprint bits")


    print("\n=== Statistical analysis of the best model (test set) ===")
    resid = yte - pred
    print("Residual normality:")
    normality(resid, "residuals")
    t, p = stats.ttest_1samp(resid, 0.0)
    print(f"  One-sample t-test, mean residual={resid.mean():.3f} vs 0: t={t:.2f}, p={p:.3g}")
    t, p = stats.ttest_rel((yte - pred) ** 2, (yte - base_pred) ** 2)
    print(f"  Paired t-test on squared errors, model vs baseline: t={t:.2f}, p={p:.3g} "
          f"({'model significantly better' if (p < ALPHA and t < 0) else 'no significant improvement'})")
    r, p = stats.pearsonr(yte, pred)
    print(f"  Pearson correlation, experimental vs predicted: r={r:.3f}, p={p:.3g}")


    fig, ax = plt.subplots(1, 2, figsize=(10, 4.5))
    ptr = train_preds[best_name]
    ax[0].scatter(ytr, ptr, s=8, alpha=0.25, color="grey", label=f"train (RMSE {np.sqrt(mean_squared_error(ytr, ptr)):.2f})")
    ax[0].scatter(yte, pred, s=12, alpha=0.7, color="tab:blue", label=f"test (RMSE {np.sqrt(mean_squared_error(yte, pred)):.2f})")
    lo, hi = min(yte.min(), pred.min(), ytr.min()), max(yte.max(), pred.max(), ytr.max())
    ax[0].plot([lo, hi], [lo, hi], "k--", lw=1)             # perfect-prediction line
    ax[0].legend(loc="upper left", fontsize=8)
    ax[0].set(xlabel="Experimental logD", ylabel="Predicted logD", title=f"{best_name}: true vs predicted")
    ax[1].hist(resid, bins=30, edgecolor="k")
    ax[1].set(xlabel="Residual (exp - pred)", ylabel="Count", title="Residual distribution")
    plt.tight_layout()
    plt.savefig("lipo_results.png", dpi=150)
    print("\nSaved plot: lipo_results.png")


if __name__ == "__main__":
    main()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.1/38.1 MB 36.9 MB/s eta 0:00:00
4200 molecules | 34 expert descriptors + 1024 fingerprint bits

scaffold split: train=3570  test=630

=== Statistical analysis (training set) ===
Normality of the target:
  logD                   Shapiro-Wilk p=9.17e-27 | D'Agostino-Pearson p=6.2e-41 -> NOT normal
Descriptors passing both normality tests: 0/34
  -> most are not normal, so Spearman rank correlation is used to rank them.

Top 10 descriptors by |Spearman rho| with logD:
MolLogP             0.383
fr_COO2            -0.327
fr_COO             -0.327
NumAromaticRings    0.304
RingCount           0.259
NHOHCount          -0.198
NumHDonors         -0.188
BertzCT             0.182
MolMR               0.179
fr_NH0              0.177 

Welch t-test, logD of acids (n=417, mean=0.95) vs non-acids (n=3153, mean=2.33): t=-21.63, p=5.29e-74

After constant + redundancy (|r|>0.90) filtering: 1048 of 1058 features

=== Test-set performance (held-out scaffolds)